<a href="https://colab.research.google.com/github/iamprabhanjan/Text2Motion/blob/main/Text_To_Video_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install "jax[cuda12_pip]==0.4.23" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

In [ ]:
!pip install diffusers==0.19.0 transformers accelerate imageio[ffmpeg] -U
!pip install -q torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 torchtext==0.14.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu116 -U

In [ ]:
!pip install torchvision==0.14.1+cu116 --extra-index-url https://download.pytorch.org/whl/cu116


In [ ]:
import torch.fx.experimental.symbolic_shapes as sym_shapes
print(dir(sym_shapes))


In [ ]:
!pip install --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html # Upgrade JAX

# Make sure the diffusers version is compatible with the newly updated JAX
!pip install diffusers transformers accelerate imageio[ffmpeg] -U
!pip install -q torch torchvision torchaudio torchtext torchdata --index-url https://download.pytorch.org/whl/cu116 -U

In [ ]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode
import datetime

In [ ]:
pipe = DiffusionPipeline.from_pretrained("iamprabhanjan/Mini-Project-ttv", torch_dtype=torch.float16, variant="fp16")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

!mkdir /content/videos

In [ ]:
import imageio
import numpy as np


In [ ]:
!nvidia-smi

In [ ]:
prompt = 'cat riding a skateboard' #@param {type:"string"}
negative_prompt = "low quality" #@param {type:"string"}
num_frames = 60 #@param {type:"raw"}
video_frames = pipe(prompt, negative_prompt=negative_prompt, num_inference_steps=25, num_frames=num_frames).frames
video_frames_reshaped = [np.array(frame) for frame in video_frames[0]]
output_video_path = export_to_video(video_frames_reshaped)

new_video_path = f'/content/videos/{datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")}.mp4'
!ffmpeg -y -i {output_video_path} -c:v libx264 -c:a aac -strict -2 {new_video_path} >/dev/null 2>&1

print(output_video_path, '->', new_video_path)

In [ ]:
!cp {new_video_path} /content/videos/tmp.mp4
mp4 = open('/content/videos/tmp.mp4','rb').read()

decoded_vid = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=400 controls><source src="{decoded_vid}" type="video/mp4"></video>')